# Bose2 — Free no-GPU cinematic crochet montage

Create a narrated 1080p crochet lesson from your own or properly licensed footage clips, with Egyptian Arabic AI narration and Arabic captions. This is a CPU/FFmpeg montage workflow: **no GPU, paid API, billing, face/avatar, or automatic publishing**. It does not generate new live-action footage; it edits clips you provide.

Keep the notebook open while rendering. The free runtime can disconnect. Edge TTS requires an internet connection and may be rate-limited. Review the final MP4 and caption timing before sharing.

In [ ]:
# Install free local video tools and speech client.
!git clone --depth 1 --branch feature/real-hands-footage-mode https://github.com/bosesamer-code/Bose2.git /content/Bose2
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-core >/dev/null
!python -m pip -q install edge-tts
import sys, subprocess
from pathlib import Path
sys.path.insert(0, '/content/Bose2')
from src.media_providers import MediaRequest
from src.free_media_providers import FreeLocalAudioProvider
from src.real_footage_providers import RealFootageVideoProvider
print('Setup complete. No GPU or paid API is used by this notebook.')

## 1. Upload your footage

Upload one or more MP4/MOV/MKV/WEBM clips that you own or have permission to use. For crochet teaching, use close-ups that clearly show the hands, hook, yarn, and stitch. Do not upload private or sensitive footage.

In [ ]:
from google.colab import files
uploaded = files.upload()
input_dir = Path('/content/bose2_cinematic_inputs')
input_dir.mkdir(parents=True, exist_ok=True)
saved = []
for name, data in uploaded.items():
    path = input_dir / Path(name).name
    path.write_bytes(data)
    saved.append(path)
video_exts = {'.mp4', '.mov', '.mkv', '.webm'}
clips = [p for p in saved if p.suffix.lower() in video_exts and p.stat().st_size > 0]
if not clips:
    raise ValueError('Upload at least one non-empty video clip (MP4/MOV/MKV/WEBM).')
print(f'Loaded {len(clips)} video clip(s):')
for clip in clips: print(' -', clip.name, f'({clip.stat().st_size:,} bytes)')
print('Only use footage you own or are licensed to use.')

## 2. Add the Egyptian Arabic narration and captions

Edit the example script and scene captions below. Captions should summarize the matching spoken section. Scene durations are approximate and are automatically scaled to fit the generated narration, so inspect the finished video and adjust timings/text if needed.

In [ ]:
script = '''أهلاً بيكم! النهارده هنتعلم خطوة كروشيه بسيطة. ثبتي الخيط على الإبرة، وبعد كده اسحبي الخيط بهدوء من الحلقة. كرري الحركة بنفس الطريقة، وخلي الغرز متساوية من غير شد زيادة. في النهاية راجعي شكل الغرز، ولو الدرس عجبك احفظيه وشاركيه مع اللي بيحبوا الكروشيه.'''
scene_plan = [
    {'scene': 1, 'purpose': 'hook', 'duration_seconds': 8, 'text': 'درس كروشيه بسيط للمبتدئين'},
    {'scene': 2, 'purpose': 'lesson', 'duration_seconds': 14, 'text': 'ثبتي الخيط على الإبرة'},
    {'scene': 3, 'purpose': 'application', 'duration_seconds': 18, 'text': 'اسحبي الخيط بهدوء من الحلقة'},
    {'scene': 4, 'purpose': 'application', 'duration_seconds': 18, 'text': 'كرري الحركة وخلي الغرز متساوية'},
    {'scene': 5, 'purpose': 'cta', 'duration_seconds': 10, 'text': 'احفظي الدرس وشاركيه مع محبي الكروشيه'},
]
if not script.strip(): raise ValueError('Write the narration script before continuing.')
print('Narration characters:', len(script))
print('Scenes/caption cards:', len(scene_plan))

In [ ]:
# Generate genuine Egyptian Arabic speech. If this fails, stop and inspect the error; no silent audio fallback is used.
job_dir = Path('/content/bose2_cinematic_job')
audio_result = FreeLocalAudioProvider().generate(
    MediaRequest('crochet-cinematic-trial', job_dir / 'audio', {'script': script, 'voice': 'ar-EG-SalmaNeural'})
)
print('Audio status:', audio_result.status)
if audio_result.status != 'completed':
    raise RuntimeError(audio_result.error or 'Egyptian Arabic narration generation failed.')
voice_path = Path(audio_result.artifacts[0])
print('Narration file:', voice_path, f'({voice_path.stat().st_size:,} bytes)')
payload = {'real_footage_paths': [str(p) for p in clips], 'audio_path': str(voice_path), 'scene_plan': scene_plan}
video_result = RealFootageVideoProvider().generate(
    MediaRequest('crochet-cinematic-trial', job_dir / 'video', payload)
)
print('Video status:', video_result.status)
if video_result.status != 'completed':
    raise RuntimeError(video_result.error or 'Video rendering failed.')
video_path = Path(video_result.artifacts[0])
print('MP4 created:', video_path, f'({video_path.stat().st_size:,} bytes)')
probe = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', str(video_path)], check=True, capture_output=True, text=True)
print('Duration (seconds):', round(float(probe.stdout.strip()), 1))

In [ ]:
# Download the MP4. Watch the full video, check speech/captions, and publish manually only after review.
files.download(str(video_path))